# Set Up

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os

# Cargar variables del archivo .env
# load_dotenv()
env_file = find_dotenv('../src/.env')
load_dotenv(env_file)

from datetime import date
from datetime import datetime

In [2]:
# credenciales
CORREO_REMITENTE=os.getenv("EMAIL_REMITENTE")
APP_PASSWORD_GMAIL=os.getenv("APP_PASSWORD_GMAIL")
CORREO_DESTINATARIO=os.getenv("EMAIL_DESTINATARIO")

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")

WHATSAPP_API_TOKEN=os.getenv("WHATSAPP_API_TOKEN")
PHONE_NUMBER_ID=os.getenv("PHONE_NUMBER_ID")
NUMERO_ASESOR=os.getenv("NUMERO_ASESOR")
NOMBRE_ASESOR=os.getenv("NOMBRE_ASESOR")

GOOGLE_SHEETS_ID=os.getenv("GOOGLE_SHEETS_ID")
GOOGLE_SHEETS_NAME=os.getenv("GOOGLE_SHEETS_NAME")

In [3]:
# Cliente de OpenAI

from openai import OpenAI

# Comprobar si la clave está cargada correctamente
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("La clave OPENAI_API_KEY no está configurada en el archivo .env.")
print("API Key cargada correctamente.")

# client = OpenAI()
client = OpenAI(api_key=OPENAI_API_KEY)

API Key cargada correctamente.


# Implementación: Gmail

In [4]:
from email.message import EmailMessage
import smtplib

# Conexión a email
def enviar_correo(nombre_lead, correo_lead, mensaje_para_lead):
  try:
    remitente = CORREO_REMITENTE
    destinatario = correo_lead
    mensaje = mensaje_para_lead

    email = EmailMessage()
    email["From"] = remitente
    email["To"] = destinatario
    email["Subject"] = "Mensaje importante para ti " + nombre_lead
    email.set_content(mensaje)

    smtp = smtplib.SMTP_SSL("smtp.gmail.com")
    smtp.login(remitente, APP_PASSWORD_GMAIL)
    smtp.sendmail(remitente, destinatario, email.as_string())
    smtp.quit()
    return True
  
  except:
    return False

In [5]:
# Invocamos la función

enviar_correo(
    nombre_lead = "Persona 1", 
    correo_lead = CORREO_DESTINATARIO, 
    mensaje_para_lead = "Bienvenid@"
)

True

# Implementación: Whatsapp

In [6]:
from heyoo import WhatsApp

def enviar_whatsapp(numero_whatsapp_asesor, mensaje_asesor):
  """para enviar mensaje a a whatsapp"""
  try:
    messenger = WhatsApp(
      token           = WHATSAPP_API_TOKEN,
      phone_number_id = PHONE_NUMBER_ID
    )
    # For sending a Text messages
    messenger.send_message(message=mensaje_asesor, recipient_id=numero_whatsapp_asesor)
    return True
  except:
    return False

In [7]:
enviar_whatsapp(
    numero_whatsapp_asesor=NUMERO_ASESOR,
    mensaje_asesor="Hola, hay un lead interesado ... y es real"
)

True

# Implementación: Google Sheet

In [ ]:
import pygsheets
import pandas as pd

# conexion a google sheets
def registrar_google_sheets(nombre_lead, correo_lead, producto_de_interes, celular_lead):
  # obtener los datos de google sheets
  url=f"https://docs.google.com/spreadsheets/d/{GOOGLE_SHEETS_ID}/gviz/tq?tqx=out:csv&sheet={GOOGLE_SHEETS_NAME}"
    
  # añadimos el nuevo registro al dataframe
  df = pd.read_csv(url)
  print('Base Inicial')
  print(df)
  print('\n')

  # df = agregar_cliente(df, nombre_lead, correo_lead, producto_de_interes, celular_lead)

  # Generar el nuevo ID
  if len(df) == 0:
      nuevo_id = 1  # Si el DataFrame está vacío, empezamos con 1
  else:
      # Tomamos el máximo ID existente y le sumamos 1
      nuevo_id = df['ID'].astype(int).max() + 1
  
  # Generar Fecha de Registro
  fecha_registro = datetime.today()
  
  # Agregar el nuevo registro
  df.loc[len(df.index)] = [fecha_registro, nuevo_id, 
    nombre_lead, correo_lead, producto_de_interes, celular_lead]
  
  print('Base Actualizada')
  print(df)

  try:
    # cargamos a google sheets
    service_account_path='../src/agr-asistente-openai.json'       # Actualizar
    gc = pygsheets.authorize(service_file=service_account_path)

    # open the google spreadsheet (where 'PY to Gsheet Test' is the name of my sheet)
    sh = gc.open_by_url(url)

    # select the first sheet
    wks = sh[0]
    # update the first sheet with df, starting at cell B2.
    wks.set_dataframe(df, (1,1)) # fila columna
    return True
  except TypeError:
    print(TypeError)
    return False

In [ ]:
registrar_google_sheets(
    nombre_lead="Persona 2",
    correo_lead=CORREO_DESTINATARIO, # "persona2@gmail.com",
    producto_de_interes="MLE",
    celular_lead="123456789"
)

# Open AI Function

In [ ]:
tools_list=[
    {
    "type": "function",
    "function": {
      "name": "registrar_google_sheets",
      "description": "Esta herramienta servirá para registrar al nuevo lead en google sheets",
      "parameters": {
        "type": "object",
        "properties": {
          "nombre_lead":{
              "type":"string",
              "description":"El nombre del lead interesado"
          },
          "correo_lead":{
              "type":"string",
              "description":"correo del lead interesado"
          },
          "producto_de_interes":{
              "type":"string",
              "description":"Producto del cual el lead está interesado"
          },
          "celular_lead":{
              "type":"string",
              "description":"Número de celular del lead"
          }
        },
        "required": ["nombre_lead","correo_lead","producto_de_interes","celular_lead"]
      }
    }
},
    {
    "type": "function",
    "function": {
      "name": "enviar_correo",
      "description": "esta función es utilizado para enviar un correo electrónico al lead interesado informándole que un asesor se contactará con él",
      "parameters": {
        "type": "object",
        "properties": {
          "nombre_lead":{
              "type":"string",
              "description":"El nombre del lead interesado"
          },
          "correo_lead":{
              "type":"string",
              "description":"correo del lead interesado"
          },
          "mensaje_para_lead":{
              "type":"string",
              "description":"Mensaje generado por el asistente para enviar al lead por correo electrónico"
          }
        },
        "required": ["nombre_lead","correo_lead","mensaje_para_lead"]
      }
    }
},
     {
    "type": "function",
    "function": {
      "name": "enviar_whatsapp",
      "description": "Esta funcion sirve para enviarle un mensaje de whatsapp al asesor indicándole se ha agregado un interesado a la base de datos",
      "parameters": {
        "type": "object",
        "properties": {
          "numero_whatsapp_asesor":{
              "type":"string",
              "description":"número de whatsapp del asesor elegido para atender al lead interesado"
          },
          "mensaje_asesor":{
              "type":"string",
              "description":"Mensaje generado por el asistente para el asesor"
          }
        },
        "required": ["numero_whatsapp_asesor","mensaje_asesor"]
      }
    }
},
 {"type":"file_search"}
]

# Open AI Asistente

In [ ]:
# Para repo
# En web actualizar
## NOMBRE_EMPRESA
## NOMBRE_ASESOR
## NUMERO_ASESOR

assistente = client.beta.assistants.create(
    name="Open AI Asistente - AGR",
    instructions="""
    Eres un asistente de ventas de NOMBRE_EMPRESA. Me ayudarás a atender las consultas de los clientes interesados en algún producto. Debes seguir las siguientes instrucciones:

    1. Tono de Respuesta:

      - Responde a los interesados con un tono amigable y profesional.
      - Usa la información proporcionada en la base de conocimiento para responder a sus consultas.

    2. Solicitud de Contacto con un Asesor de Ventas:

      - Si un interesado solicita contactarse con un asesor de ventas para más detalles, ejecuta la función 'registrar_google_sheets'.
      - Para ejecutar esta función, necesitas pedir y registrar los siguientes datos del interesado:
          - Nombre completo
          - Correo electrónico
          - Programa de interés
          - Celular

      - La función 'registrar_google_sheets' devolverá 'true' si el registro es exitoso, y 'false' si el correo electrónico no existe o no está validado.

    3. Contacto con el Asesor de Ventas:

      - Si 'registrar_google_sheets' devuelve true, envía un mensaje al asesor de ventas NOMBRE_ASESOR al número de WhatsApp NUMERO_ASESOR utilizando la función 'enviar_whatsapp'.
      - El mensaje debe mencionar que hay un posible comprador interesado y que debe ser contactado urgentemente. Incluye el nombre del interesado y su correo electrónico en el mensaje.

    4. Confirmación al Interesado:

      - Si el registro del interesado en la base de datos se completa satisfactoriamente ('registrar_google_sheets' devuelve true), envía un correo electrónico al interesado utilizando la función 'enviar_correo'.
      - En el correo, menciona que un asesor de ventas se contactará muy pronto con él. Proporciona también información detallada del producto de interés.
      - Recuerda ser claro y preciso en todos los mensajes y asegurar que toda la información relevante sea comunicada adecuadamente.

    """,
    model="gpt-4-turbo-preview",      # Actualizar por modelo más potente, tener en cuenta costos
    tools=tools_list,

    #tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
)

In [ ]:
# Probar asistente en Dashboard de Open AI: https://platform.openai.com/ -> Playground
# Verificar que se hayan creado las funciones
# Attach documents adicionales (File Research)

# Open AI Asistente VSC

In [ ]:
# Opcional

In [ ]:
from time import sleep
import json
def run_excecuter(run):

  while True:
    run_status=client.beta.threads.runs.retrieve(
        thread_id=run.thread_id,
        run_id=run.id
    )

    if run_status.status =="completed":
      print("accion terminada") #FUNCIONA PARA NUESTRO VSCODE O LA TERMINAL
      break
    
    elif run_status.status=="requires_action":
      print("requiere accion")

      list_of_actions=run_status.required_action.submit_tool_outputs.tool_calls
      print("-----"*20)
      print(list_of_actions)
      print("-----"*20)

      tools_output_list=[]

      for accion in list_of_actions:

        if accion.function.name == "registrar_google_sheets":
          nombre=accion.function.name
          argumentos=json.loads(accion.function.arguments)

          print("Nombre de la funcion a ejecutar: ", nombre)
          print("Argumentos de la función: ", argumentos)

          interesado_agregado=registrar_google_sheets(
            argumentos["nombre_lead"],
            argumentos["correo_lead"],
            argumentos["producto_de_interes"], 
            argumentos["celular_lead"])

          tools_output_list.append(
              {
                  "tool_call_id": accion.id,
                  "output": str(interesado_agregado)
              }
          )

        elif accion.function.name =="enviar_correo":

          nombre=accion.function.name
          argumentos=json.loads(accion.function.arguments)

          print("Nombre de la funcion a ejecutar: ", nombre)
          print("Argumentos de la funcion: ", argumentos)

          correo_enviado=enviar_correo(argumentos["nombre_lead"], argumentos["correo_lead"], argumentos["mensaje_para_lead"])

          tools_output_list.append(
              {
                  "tool_call_id": accion.id,
                  "output": str(correo_enviado)
              }
          )

        elif accion.function.name =="enviar_whatsapp":

          nombre=accion.function.name
          argumentos=json.loads(accion.function.arguments)

          print("Nombre de la funcion a ejecutar: ", nombre)
          print("Argumentos de la funcion: ", argumentos)

          whatsapp_enviado=enviar_whatsapp(
            numero_whatsapp_asesor=argumentos["numero_whatsapp_asesor"], 
            mensaje_asesor=argumentos["mensaje_asesor"])

          tools_output_list.append(
              {
                  "tool_call_id": accion.id,
                  "output": str(whatsapp_enviado)
              }
          )

        else:
          return "No se encontró la accion"
      
      print("ejecucion de acciones ha terminado")
      print(tools_output_list)
      client.beta.threads.runs.submit_tool_outputs(
          thread_id=run.thread_id,
          run_id=run.id,
          tool_outputs=tools_output_list
      )

    else:
      print("Esperando respuesta del Asistente")
      sleep(3)

# []